# OpenCV vision basics

Classical computer vision with OpenCV 5 from `opencv-contrib-python-headless`. The notebook runs offline on generated fixtures, so it needs no optional Atlas service. Images render through Matplotlib: the image has no GUI toolkit, and `cv2.imshow` is not used (#601).


In [ ]:
import importlib.metadata as metadata

import cv2
import matplotlib.pyplot as plt
import numpy as np

distribution = metadata.packages_distributions()["cv2"]
gui = next(
    line.split(":", 1)[1].strip()
    for line in cv2.getBuildInformation().splitlines()
    if line.strip().startswith("GUI:")
)
print("distribution:", distribution)
print("cv2:", cv2.__version__, "| numpy:", np.__version__, "| GUI:", gui)
assert distribution == ["opencv-contrib-python-headless"]
assert cv2.__version__.startswith("5.0")


## Deterministic fixture

A red square and a blue circle on a dark background, with small seeded noise so thresholds are exercised. Every run produces the same pixels.


In [ ]:
rng = np.random.default_rng(601)
bgr = np.zeros((120, 160, 3), dtype=np.uint8)
cv2.rectangle(bgr, (20, 30), (90, 100), (0, 0, 255), thickness=-1)  # red, in BGR order
cv2.circle(bgr, (120, 60), 25, (255, 0, 0), thickness=-1)           # blue, in BGR order
bgr = np.clip(bgr.astype(np.int16) + rng.integers(0, 8, size=bgr.shape), 0, 255).astype(np.uint8)
assert bgr.shape == (120, 160, 3) and bgr.dtype == np.uint8


## Color conversion

OpenCV stores images as BGR. Converting to RGB swaps the first and last channels.

In [ ]:
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
assert rgb.dtype == np.uint8 and rgb.shape == bgr.shape
assert np.array_equal(rgb[..., 0], bgr[..., 2])


## Resize

`INTER_AREA` is the usual choice for shrinking.

In [ ]:
small = cv2.resize(bgr, (80, 60), interpolation=cv2.INTER_AREA)
assert small.shape == (60, 80, 3) and small.dtype == np.uint8


## Perspective warp

A four-point homography maps the frame onto a skewed quadrilateral.

In [ ]:
src = np.float32([[0, 0], [159, 0], [159, 119], [0, 119]])
dst = np.float32([[10, 0], [150, 10], [140, 119], [0, 110]])
matrix = cv2.getPerspectiveTransform(src, dst)
warped = cv2.warpPerspective(bgr, matrix, (160, 120))
assert warped.shape == bgr.shape and warped.dtype == np.uint8


## Threshold and contours

A low threshold keeps both shapes; the noise stays below it.

In [ ]:
gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
_, mask = cv2.threshold(gray, 20, 255, cv2.THRESH_BINARY)
contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
assert mask.dtype == np.uint8 and mask.shape == gray.shape
assert len(contours) == 2


## Feature extraction

ORB finds corners and gives 32-byte binary descriptors.

In [ ]:
orb = cv2.ORB_create(nfeatures=200)
keypoints, descriptors = orb.detectAndCompute(gray, None)
assert len(keypoints) > 0
assert descriptors.dtype == np.uint8 and descriptors.shape == (len(keypoints), 32)


## Contrib-only API

`cv2.xfeatures2d` ships only in the contrib build. LATCH computes binary descriptors at the ORB keypoints and needs no downloaded data. BoostDesc needs downloaded data that the wheel does not include, so it is not used here.


In [ ]:
latch = cv2.xfeatures2d.LATCH_create()
kept, latch_descriptors = latch.compute(gray, keypoints)
assert 0 < len(kept) <= len(keypoints)
assert latch_descriptors.dtype == np.uint8 and latch_descriptors.shape == (len(kept), 32)


## Display

Matplotlib renders the frames. Nothing opens a window.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for axis, image, title in zip(
    axes, [rgb, small, warped, mask], ["RGB", "resized 80x60", "warped", "threshold"]
):
    axis.imshow(image, cmap="gray" if image.ndim == 2 else None)
    axis.set_title(title)
    axis.axis("off")
plt.show()


## Summary

Each step asserts its output shape and dtype. Optional Atlas services are not used, so nothing here waits on a network call.